# Óptimo y consistencia de los parámetros

Vista del barrido del notebook `01`: **no vuelve a correr el modelo** salvo para
la especie que se grafica en detalle (celda 4, una sola combinación).

  * ROC y distribución de scores de la especie elegida;
  * efecto de cada parámetro sobre la AUC01 (**α**, β, λ, γ);
  * **β = 0 vs β > 0** — G′r vs G′rk. Como la grilla ya tiene β = 0, esta
    comparación es una *vista* del barrido y no un análisis aparte (README §1.5);
  * plateau del óptimo: cuánto se pierde al no elegir el argmax exacto;
  * consistencia entre especies y entre grupos taxonómicos.

Salidas: `02_optimos_por_especie.csv`, `02_consistencia.csv`, `02_plateau.csv`.

## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

V4 = "/home/ggiordano/TDR/TDR_2026_v4"
sys.path.insert(0, f"{V4}/comun")
sys.path.insert(0, f"{V4}/genome_prioritization")
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_genome as fg              # el .py de esta carpeta

SALIDAS = tdr.out("genome_prioritization")
FIGURAS = SALIDAS / "figuras"
NB      = "02"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

## Datos

In [ ]:
BARRIDO = tdr.out("genome_prioritization")             # salidas del notebook 01
barrido = fg.cargar_barrido(BARRIDO, "01")
datos   = tdr.cargar_db(anotaciones=True, cluster_consistent=False)
barrido.shape, tdr.leer_meta(BARRIDO, "01")["fecha"]

## Acondicionamiento

In [ ]:
spout = tdr.SP_FOCO                                      # el protozoo parásito que se mira en detalle
optimos = fg.optimos_por_especie(barrido)
alpha_val, beta_val, lam_val, gamma_val = optimos.loc[
    optimos["especie"] == spout, ["alpha", "beta", "lambda_", "gamma"]].values[0]
lam_val = None if pd.isna(lam_val) else lam_val         # NaN = modo híbrido

print(f"{spout}: alpha={alpha_val} beta={beta_val} lambda={lam_val} gamma={gamma_val}")
optimos[["especie", "AUC01", "AUC", "alpha", "beta", "lambda_", "gamma"]]

## Corrida

In [ ]:
# celda de corrida: una sola combinación (la óptima de spout) para poder graficar la ROC
ctx = fg.preparar_especie(datos, spout)
cat_rs = tdr.relevance_scores(alpha=alpha_val, pv=ctx["cat_rs"])
rnk = tdr.propagar(datos.sta, ctx["seed"], cat_rs, beta=beta_val, lambda_=lam_val, gamma=gamma_val)
rnk["target_id"] = rnk["target_id"].astype(str)
rnk["druggable"] = rnk["target_id"].isin(ctx["tp"]).astype(int)
sp_rnk = tdr.ranking_especie(rnk, ctx["sp_targets"], ctx["tp"])

frecuencias, spearman = fg.consistencia(barrido, k=10)
pl = fg.plateau(barrido)

optimos.to_csv(SALIDAS / f"{NB}_optimos_por_especie.csv", index=False)
frecuencias.to_csv(SALIDAS / f"{NB}_consistencia.csv", index=False)
spearman.to_csv(SALIDAS / f"{NB}_spearman_especies.csv")
pl.to_csv(SALIDAS / f"{NB}_plateau.csv", index=False)
fg.escribir_meta(SALIDAS, NB, notebook="02_optimo_y_consistencia.ipynb",
                 params=fg.PARAMS, especie_detalle=spout, barrido_nb="01")

# Resultados

In [ ]:
optimos     = pd.read_csv(SALIDAS / f"{NB}_optimos_por_especie.csv")
frecuencias = pd.read_csv(SALIDAS / f"{NB}_consistencia.csv")
spearman    = pd.read_csv(SALIDAS / f"{NB}_spearman_especies.csv", index_col=0)
pl          = pd.read_csv(SALIDAS / f"{NB}_plateau.csv")
pl

In [ ]:
fig = fg.fig_roc(sp_rnk, rnk, spout, (alpha_val, beta_val, lam_val, gamma_val), plt)
tdr.guardar(fig, f"{NB}_f01_roc_{spout}", FIGURAS)

In [ ]:
# Efecto de cada parámetro; el panel de beta es G'r (beta=0) vs G'rk (beta>0)
fig = fg.fig_grilla_beta(barrido, plt)
tdr.guardar(fig, f"{NB}_f02_grilla_beta", FIGURAS)

In [ ]:
fig = fg.fig_plateau(pl, plt)
tdr.guardar(fig, f"{NB}_f03_plateau", FIGURAS)

In [ ]:
fig = fg.fig_consistencia(spearman, plt)
tdr.guardar(fig, f"{NB}_f04_consistencia_especies", FIGURAS)

In [ ]:
# Enriquecimiento de cada valor de parámetro en el top-10: el factor dominante
fig, axes = plt.subplots(1, 4, figsize=(11, 3), sharey=True, tight_layout=True)
for ax, col in zip(axes, ["alpha", "beta", "lambda_", "gamma"]):
    f = frecuencias[frecuencias["parametro"] == col].sort_values("valor")
    ax.bar(f["valor"].astype(str), f["enriquecimiento"], color=tdr.S1)
    ax.axhline(1, color=tdr.MUTED, ls="--", lw=1)
    ax.set_title(col, fontsize=9)
axes[0].set_ylabel("enriquecimiento en el top-10")
tdr.guardar(fig, f"{NB}_f05_enriquecimiento_topk", FIGURAS)